<a href="https://colab.research.google.com/github/OmaRecalde/machinelearning2026/blob/main/API_Arquitectura_Deteccion_Cancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avance: Arquitectura de una API para la Detección de Cáncer de Mama

## Neurona de McCulloch y Pitts + FastAPI

**Objetivo:** diseñar una API que consuma el modelo de la neurona MP entrenado con el dataset UCI ML Breast Cancer Wisconsin, para realizar predicciones (Benigno / Maligno) usando únicamente las variables que realmente afectan la predicción.

**Alcance de este avance:**
1. Definir la arquitectura de la solución.
2. Seleccionar las variables más relevantes (AUC individual).
3. Entrenar la neurona MP solo con esas variables.
4. Guardar el modelo para que la API lo consuma.
5. Construir un prototipo de la API con FastAPI y probarlo.

## 1. Arquitectura propuesta

La solución se divide en dos etapas: **entrenamiento** (se ejecuta una vez) y **servicio** (la API que responde predicciones).

    ┌──────────────────────── ETAPA 1: ENTRENAMIENTO (Notebook) ────────────────────────┐
    │                                                                                     │
    │  Dataset UCI  →  Selección de variables  →  Binarización  →  Entrenamiento MP      │
    │  (569 x 30)       (Top 4 por AUC)            (punto de corte)  (threshold óptimo)   │
    │                                                                    │                │
    │                                                                    ▼                │
    │                                                          modelo_mp.json             │
    │                                                (variables, cortes, threshold)       │
    └─────────────────────────────────────────────────────────────────────────────────────┘
                                                                         │
                                                                         ▼
    ┌──────────────────────────── ETAPA 2: SERVICIO (FastAPI) ───────────────────────────┐
    │                                                                                     │
    │  Cliente (web / app / Postman)                                                      │
    │        │  POST /predict  { 4 valores del paciente }                                 │
    │        ▼                                                                            │
    │  Validación (Pydantic)  →  Binarización  →  Neurona MP  →  Respuesta JSON           │
    │                                                            (Benigno / Maligno)      │
    └─────────────────────────────────────────────────────────────────────────────────────┘

### Endpoints de la API

| Método | Ruta | Descripción |
|---|---|---|
| GET | `/` | Verifica que la API esté funcionando |
| GET | `/modelo` | Muestra las variables, puntos de corte y threshold del modelo |
| POST | `/predict` | Recibe los 4 valores del paciente y devuelve la predicción |

### Estructura del proyecto

    api-cancer-mp/
    ├── notebook_entrenamiento.ipynb   # Este notebook
    ├── modelo_mp.json                 # Modelo entrenado (artefacto)
    ├── main.py                        # API FastAPI
    └── requirements.txt               # fastapi, uvicorn, pydantic

### Tecnologías

- **Python + scikit-learn + pandas:** carga de datos, selección de variables y métricas.
- **Neurona MP (implementación propia):** modelo de clasificación.
- **FastAPI + Pydantic:** construcción de la API y validación de datos de entrada.
- **Uvicorn:** servidor para ejecutar la API.

## 2. Lectura de datos

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

cancer_name = load_breast_cancer()
X = cancer_name.data
Y = cancer_name.target  # 0 = Maligno, 1 = Benigno

df = pd.DataFrame(X, columns=cancer_name.feature_names)
print("Dimensiones:", df.shape)
print("Clases:", dict(zip(cancer_name.target_names, np.bincount(Y))))

Dimensiones: (569, 30)
Clases: {np.str_('malignant'): np.int64(212), np.str_('benign'): np.int64(357)}


## 3. Selección de variables (AUC individual)

Para que la API solo pida los datos que realmente afectan la predicción, se calcula el **AUC individual** de cada variable. Un AUC cercano a 1 indica que la variable, por sí sola, separa muy bien los casos benignos de los malignos.

In [2]:
from sklearn.metrics import roc_auc_score

auc_individual = {}
for columna in df.columns:
    auc = roc_auc_score(Y, df[columna])
    # Si la variable se relaciona de forma inversa, se toma 1 - AUC
    auc_individual[columna] = max(auc, 1 - auc)

ranking = (pd.DataFrame(list(auc_individual.items()), columns=["Variable", "AUC individual"])
             .sort_values("AUC individual", ascending=False)
             .reset_index(drop=True))
ranking.index = ranking.index + 1
ranking.head(10).round(3)

,Variable,AUC individual
1,worst perimeter,0.975
2,worst radius,0.970
3,worst area,0.970
4,worst concave points,0.967
5,mean concave points,0.964
6,mean perimeter,0.947
7,mean area,0.938
8,mean concavity,0.938
9,mean radius,0.938
10,area error,0.926


Según el ranking, se seleccionan las **4 variables clave**:

| # | Variable | Descripción |
|---|---|---|
| 1 | worst perimeter | Peor perímetro del núcleo celular |
| 2 | worst radius | Peor radio del núcleo celular |
| 3 | worst area | Peor área del núcleo celular |
| 4 | worst concave points | Peor cantidad de puntos cóncavos del contorno |

Usar pocas variables simplifica la API (el usuario solo envía 4 valores) y evita que el modelo memorice datos irrelevantes. Un resultado del 100% indicaría memorización, no aprendizaje.

In [4]:
VARIABLES = ["worst perimeter", "worst radius", "worst area", "worst concave points"]

X_sel = df[VARIABLES]
X_sel.head()

,worst perimeter,worst radius,worst area,worst concave points
0,184.60,25.38,2019.0,0.2654
1,158.80,24.99,1956.0,0.1860
2,152.50,23.57,1709.0,0.2430
3,98.87,14.91,567.7,0.2575
4,152.20,22.54,1575.0,0.1625


## 4. División del conjunto de datos

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_sel, Y, stratify=Y, random_state=42)
print("Tamaño de entrenamiento:", len(X_train))
print("Tamaño de pruebas:", len(X_test))

Tamaño de entrenamiento: 426
Tamaño de pruebas: 143


## 5. Binarización con puntos de corte

La neurona MP solo procesa entradas binarias. En el ejercicio anterior se usó `pd.cut(bins=2)`, que divide cada variable en el punto medio entre su mínimo y su máximo.

Para la API se **guarda ese punto de corte**, porque cuando llegue un paciente nuevo hay que binarizar su valor con el mismo criterio del entrenamiento:

- Valor ≤ punto de corte → **1**
- Valor > punto de corte → **0**

In [6]:
# Punto de corte = punto medio entre el mínimo y el máximo (equivalente a pd.cut con bins=2)
cortes = {v: float((X_train[v].min() + X_train[v].max()) / 2) for v in VARIABLES}

def binarizar_df(X, cortes):
    return pd.DataFrame({v: (X[v] <= cortes[v]).astype(int) for v in cortes}, index=X.index)

X_train_bin = binarizar_df(X_train, cortes)
X_test_bin = binarizar_df(X_test, cortes)

print("Puntos de corte:", {k: round(c, 4) for k, c in cortes.items()})
X_train_bin.head()

Puntos de corte: {'worst perimeter': 139.855, 'worst radius': 20.53, 'worst area': 1808.6, 'worst concave points': 0.1455}


,worst perimeter,worst radius,worst area,worst concave points
517,0,0,1,0
287,1,1,1,1
25,0,0,1,0
253,1,1,1,0
369,0,0,0,0


## 6. Neurona MP

In [7]:
from sklearn.metrics import accuracy_score

class MPNeuron:
    def __init__(self):
        self.threshold = None

    def model(self, x):
        z = sum(x)
        return (z >= self.threshold)

    def predict(self, X):
        Y = []
        for x in X:
            Y.append(self.model(x))
        return np.array(Y)

    def fit(self, X, Y):
        accuracy = {}
        # Se prueba desde 0 hasta el número total de entradas (inclusive)
        for th in range(X.shape[1] + 1):
            self.threshold = th
            Y_pred = self.predict(X)
            accuracy[th] = accuracy_score(Y, Y_pred)
        self.threshold = max(accuracy, key=accuracy.get)

## 7. Entrenamiento y evaluación

In [8]:
mp_neurona = MPNeuron()
mp_neurona.fit(X_train_bin.to_numpy(), y_train)
print("Threshold óptimo:", mp_neurona.threshold)

Y_pred = mp_neurona.predict(X_test_bin.to_numpy())
print("Exactitud en pruebas:", round(accuracy_score(y_test, Y_pred), 4))

Threshold óptimo: 4
Exactitud en pruebas: 0.9301


## 8. Guardar el modelo para la API

La API no necesita volver a entrenar. Solo necesita tres cosas: las variables, los puntos de corte y el threshold. Se guardan en un archivo `modelo_mp.json`.

In [9]:
import json

modelo = {
    "features": VARIABLES,
    "cortes": cortes,
    "threshold": int(mp_neurona.threshold),
}

with open("modelo_mp.json", "w", encoding="utf-8") as f:
    json.dump(modelo, f, indent=4, ensure_ascii=False)

print(json.dumps(modelo, indent=4, ensure_ascii=False))

{
    "features": [
        "worst perimeter",
        "worst radius",
        "worst area",
        "worst concave points"
    ],
    "cortes": {
        "worst perimeter": 139.85500000000002,
        "worst radius": 20.53,
        "worst area": 1808.6,
        "worst concave points": 0.1455
    },
    "threshold": 4
}


## 9. Prototipo de la API con FastAPI

In [10]:
!pip install fastapi uvicorn httpx -q

In [11]:
%%writefile main.py
import json
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Cargar el modelo entrenado
with open("modelo_mp.json", "r", encoding="utf-8") as f:
    MODELO = json.load(f)

FEATURES = MODELO["features"]
CORTES = MODELO["cortes"]
THRESHOLD = MODELO["threshold"]

app = FastAPI(
    title="API Detección de Cáncer de Mama - Neurona MP",
    version="0.1.0",
)

# Datos de entrada del paciente (validación con Pydantic)
class Paciente(BaseModel):
    worst_perimeter: float = Field(..., gt=0, description="Peor perímetro")
    worst_radius: float = Field(..., gt=0, description="Peor radio")
    worst_area: float = Field(..., gt=0, description="Peor área")
    worst_concave_points: float = Field(..., ge=0, description="Peor puntos cóncavos")

def binarizar(nombre, valor):
    return 1 if valor <= CORTES[nombre] else 0

@app.get("/")
def inicio():
    return {"estado": "API funcionando"}

@app.get("/modelo")
def info_modelo():
    return MODELO

@app.post("/predict")
def predecir(paciente: Paciente):
    datos = paciente.model_dump()
    entradas_bin = {f: binarizar(f, datos[f.replace(" ", "_")]) for f in FEATURES}
    suma = sum(entradas_bin.values())
    clase = int(suma >= THRESHOLD)
    return {
        "prediccion": "Benigno" if clase == 1 else "Maligno",
        "clase": clase,
        "entradas_binarias": entradas_bin,
        "suma": suma,
        "threshold": THRESHOLD,
    }

Writing main.py


## 10. Pruebas de la API

Se prueba la API dentro de Colab con `TestClient`, que simula peticiones HTTP sin necesidad de levantar un servidor.

In [12]:
import importlib
import main
importlib.reload(main)
from fastapi.testclient import TestClient

client = TestClient(main.app)

print("GET / →", client.get("/").json())
print("GET /modelo →", client.get("/modelo").json())

GET / → {'estado': 'API funcionando'}
GET /modelo → {'features': ['worst perimeter', 'worst radius', 'worst area', 'worst concave points'], 'cortes': {'worst perimeter': 139.85500000000002, 'worst radius': 20.53, 'worst area': 1808.6, 'worst concave points': 0.1455}, 'threshold': 4}


In [13]:
# Paciente real del conjunto de pruebas con diagnóstico MALIGNO
fila = X_test[y_test == 0].iloc[0]
paciente = {v.replace(" ", "_"): float(fila[v]) for v in VARIABLES}

print("Datos enviados:", paciente)
client.post("/predict", json=paciente).json()

Datos enviados: {'worst_perimeter': 138.1, 'worst_radius': 21.08, 'worst_area': 1349.0, 'worst_concave_points': 0.1974}


{'prediccion': 'Maligno',
 'clase': 0,
 'entradas_binarias': {'worst perimeter': 1,
  'worst radius': 0,
  'worst area': 1,
  'worst concave points': 0},
 'suma': 2,
 'threshold': 4}

In [14]:
# Paciente real del conjunto de pruebas con diagnóstico BENIGNO
fila = X_test[y_test == 1].iloc[0]
paciente = {v.replace(" ", "_"): float(fila[v]) for v in VARIABLES}

print("Datos enviados:", paciente)
client.post("/predict", json=paciente).json()

Datos enviados: {'worst_perimeter': 93.63, 'worst_radius': 14.45, 'worst_area': 624.1, 'worst_concave_points': 0.08045}


{'prediccion': 'Benigno',
 'clase': 1,
 'entradas_binarias': {'worst perimeter': 1,
  'worst radius': 1,
  'worst area': 1,
  'worst concave points': 1},
 'suma': 4,
 'threshold': 4}

In [15]:
# Validación: si falta un dato, la API responde con error 422
client.post("/predict", json={"worst_radius": 15.0}).status_code

422

## 11. Próximos pasos

1. **Ejecutar la API localmente:** `uvicorn main:app --reload` y abrir `http://127.0.0.1:8000/docs` para la documentación automática (Swagger).
2. **Crear `requirements.txt`** con las dependencias del proyecto.
3. **Agregar un endpoint de predicción por lotes** (`/predict/batch`) para varios pacientes a la vez.
4. **Registrar las predicciones** (logs) para monitorear el uso del modelo.
5. **Contenerizar con Docker** y desplegar en un servidor o en la nube.
6. **Mejorar el modelo:** comparar la neurona MP con un perceptrón o una regresión logística usando las mismas 4 variables.

> **Nota:** esta API es un ejercicio académico y no reemplaza un diagnóstico médico.

In [16]:
import json, nbformat
from nbconvert import HTMLExporter
from google.colab import _message, files

nb = _message.blocking_request('get_ipynb')['ipynb']
nb = nbformat.reads(json.dumps(nb), as_version=4)
html, _ = HTMLExporter().from_notebook_node(nb)

with open('notebook.html', 'w', encoding='utf-8') as f:
    f.write(html)
files.download('notebook.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>